In [1]:
seed = 0
na = 30
solver_ = 'appsi_highs'
ratio = 0.3

In [2]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from torch_geometric.data import Data
import pyomo.environ as pyo
from pyomo.environ import value
import numpy as np
import pandas as pd
import random
import copy

# --- PARTIE 1 : L'ARCHITECTURE DU GNN ---
class MILPGNN(torch.nn.Module):
    def __init__(self, num_node_features, hidden_channels, num_classes):
        super(MILPGNN, self).__init__()
        self.conv1 = SAGEConv(num_node_features, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.conv3 = SAGEConv(hidden_channels, hidden_channels)
        
        # Le classifieur prédit la probabilité d'appartenir à chaque classe (temps t ou jamais)
        self.classifier = torch.nn.Sequential(
            torch.nn.Linear(hidden_channels, hidden_channels // 2),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_channels // 2, num_classes)
        )

    def forward(self, x, edge_index):
        # Passage de messages dans le graphe (pas besoin de edge_attr pour ce design)
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = self.conv3(x, edge_index)
        x = F.relu(x)
        
        out = self.classifier(x)
        return out


# --- PARTIE 2 : EXTRACTION DU GRAPHE DEPUIS PYOMO ---
def extract_graph_from_pyomo(model):
    """
    Transforme le modèle Pyomo en un graphe où les Noeuds = Sites.
    """
    node_features = []
    labels = [] 
    var_indices = [] 
    
    S_list = list(model.S)
    T_list = list(model.T)
    num_T = len(T_list)
    
    mapping = {s: idx for idx, s in enumerate(S_list)}
    
    # 1. Extraction des caractéristiques (Features) et Labels
    for s in S_list:
        # Features demandées : Degré (nb zones) et Clients potentiels totaux
        degree = len(model.As[s])
        clients = sum(pyo.value(model.u_a[a]) for a in model.As[s])
        
        node_features.append([float(degree), float(clients)])
        var_indices.append(s)
        
        # Label: A quel instant t le site s'active-t-il ?
        if model.z[T_list[0], s].value is not None:
            activation_t = num_T # Par défaut: Jamais activé
            for t in T_list:
                if round(model.z[t, s].value) == 1:
                    activation_t = t
                    break
            labels.append(activation_t)
        else:
            labels.append(-1) # Inconnu pour l'inférence
            
    # 2. Création des arêtes (Edges)
    # Deux sites sont connectés s'ils couvrent au moins une zone en commun
    edges_set = set()
    for a in model.A:
        sites_in_a = model.Sa[a]
        for i in range(len(sites_in_a)):
            for j in range(i+1, len(sites_in_a)):
                idx1, idx2 = mapping[sites_in_a[i]], mapping[sites_in_a[j]]
                edges_set.add((idx1, idx2))
                edges_set.add((idx2, idx1))
                
    edge_list = list(edges_set)
    # Sécurité si graphe totalement déconnecté (boucles sur soi-même)
    if len(edge_list) == 0:
        for idx in mapping.values():
            edge_list.append([idx, idx])

    # 3. Conversion en Tenseurs PyTorch
    x = torch.tensor(node_features, dtype=torch.float)
    
    # Normalisation des features (Crucial pour les degrés/clients qui ont des échelles différentes)
    if x.shape[0] > 0:
        x_mean = x.mean(dim=0, keepdim=True)
        x_std = x.std(dim=0, keepdim=True) + 1e-6
        x = (x - x_mean) / x_std

    edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
    y = torch.tensor(labels, dtype=torch.long) # LongTensor requis pour CrossEntropy

    data = Data(x=x, edge_index=edge_index, y=y)
    data.var_mapping = var_indices
    data.num_classes = num_T + 1 # t de 0 à Tmax, + 1 classe pour "Jamais"
    
    return data

In [3]:
# --- PARTIE 3 : ENTRAÎNEMENT DU MODÈLE ---
def train_gnn(dataset_graphs, epochs=50):
    num_classes = dataset_graphs[0].num_classes
    # num_node_features = 2 (degree, total_clients)
    gnn_model = MILPGNN(num_node_features=2, hidden_channels=64, num_classes=num_classes)
    optimizer = torch.optim.Adam(gnn_model.parameters(), lr=0.005)
    
    # Remplacement par CrossEntropy pour la classification multi-classes
    criterion = torch.nn.CrossEntropyLoss() 

    gnn_model.train()
    for epoch in range(epochs):
        total_loss = 0
        for data in dataset_graphs:
            optimizer.zero_grad()
            out = gnn_model(data.x, data.edge_index)
            
            mask = data.y != -1 
            if mask.sum() > 0:
                loss = criterion(out[mask], data.y[mask])
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
            
        if epoch % 10 == 0:
            print(f'Epoch {epoch:03d}, Loss: {total_loss / len(dataset_graphs):.4f}')
            
    return gnn_model

# --- PARTIE 4 : LA MÉTHODE NLNS (INFERENCE) ---
def solve_with_nlns(pyomo_model, trained_gnn, solver_name=solver_, fix_ratio=0.3):
    print("\n--- Démarrage du NLNS ---")
    trained_gnn.eval()
    
    data = extract_graph_from_pyomo(pyomo_model)
    
    with torch.no_grad():
        logits = trained_gnn(data.x, data.edge_index)
    
    # Transformation des logits en probabilités
    probs = F.softmax(logits, dim=1)
    
    # On prend la classe prédite et la probabilité maximale (certitude)
    certainties, predictions = torch.max(probs, dim=1)
    
    certainties = certainties.numpy()
    predictions = predictions.numpy()
    
    # Tri des variables par certitude
    n_vars_to_fix = int(len(predictions) * fix_ratio)
    indices_to_fix = np.argsort(certainties)[-n_vars_to_fix:]
    
    fixed_count = 0
    T_list = list(pyomo_model.T)
    num_T = len(T_list)
    
    for idx in indices_to_fix:
        s = data.var_mapping[idx]
        t_activation = predictions[idx] # Temps d'activation prédit
        
        # Application de la prédiction dans Pyomo
        # Si t_activation = num_T, ça veut dire "Jamais activé" -> tous les z restent à 0
        for t_idx, t in enumerate(T_list):
            val_to_fix = 1 if t_idx >= t_activation else 0
            pyomo_model.z[t, s].fix(val_to_fix)
            fixed_count += 1
            
    print(f"NLNS : {fixed_count} variables z[t,s] fixées par le GNN.")
    print(f"NLNS : Lancement du solveur classique sur les variables restantes...")
    
    solver = pyo.SolverFactory(solver_name)
    results = solver.solve(pyomo_model, tee=False)
    
    print("NLNS : Résolution terminée.")
    print("Valeur de l'objectif NLNS :", value(pyomo_model.obj))
    
    return pyomo_model

In [4]:
def build_pyomo_instance(seed_value, num_areas):
    """
    Construit une instance Pyomo unique en fonction d'une graine aléatoire.
    Ceci utilise VOTRE logique de construction de modèle.
    """
    model = pyo.ConcreteModel()
    
    # -- VOS SETS ET PARAMÈTRES (Adaptés pour la génération) --
    T = [0, 1, 2, 3, 4, 5]
    model.T = pyo.Set(initialize=T)
    
    # Sélection aléatoire des zones pour créer une instance unique
    dA = pd.read_csv("AREAS.csv", sep=";")  
    random.seed(seed_value)
    n = len(dA["AREAS"])
    
    na = min(num_areas, n) 
    A = [dA["AREAS"][i] for i in random.sample(range(0, n), na)]
    model.A = pyo.Set(initialize=A)
    
    from itertools import product
    
    I=['ORANGE','FREE MOBILE', 'BOUYGUES TELECOM','SFR']
    τ='ORANGE'
    O=['o3G-FREE MOBILE', 'o3G-BOUYGUES TELECOM', 'o3G-SFR','o3G-ORANGE',
       'o4G-FREE MOBILE','o4G-BOUYGUES TELECOM','o4G-SFR','o4G-ORANGE',
       'o5G-FREE MOBILE','o5G-BOUYGUES TELECOM','o5G-SFR','o5G-ORANGE']  
    
    O_i = {"ORANGE" : ['o3G-ORANGE' , 'o4G-ORANGE', 'o5G-ORANGE'] , 
           'FREE MOBILE' : ['o3G-FREE MOBILE', 'o4G-FREE MOBILE', 'o5G-FREE MOBILE'] ,
           'BOUYGUES TELECOM' : ['o3G-BOUYGUES TELECOM', 'o4G-BOUYGUES TELECOM', 'o5G-BOUYGUES TELECOM'] , 
           'SFR' : ['o3G-SFR', 'o4G-SFR', 'o5G-SFR']  }
    
    NG = 'o5G-ORANGE'

    model.I = pyo.Set(initialize=I)
    model.O = pyo.Set(initialize=O)
    model.O_i = O_i

    # --- Couples utiles ---
    C_space = list(product([0,1], repeat=len(I)))
    df_link = pd.read_csv("AREAS_SITES_LINK.csv", sep=";")

    Sa_dict = {}
    S = [] 

    for _, row in df_link.iterrows():
        zone = row["AREAS"]
        site = row["SITES"]
        if zone not in A:
            continue

        if zone not in Sa_dict :
            Sa_dict[zone] = []

        if site not in Sa_dict[zone]:
            Sa_dict[zone].append(site)
            
        if site not in S:
            S.append(site)

    model.S = pyo.Set(initialize=S)

    As_dict = {}
    for _, row in df_link.iterrows():
        zone = row["AREAS"]
        site = row["SITES"]
        if site not in S:
            continue

        if site not in As_dict:
            As_dict[site] = []
        if zone in A and zone not in As_dict[site]:
            As_dict[site].append(zone)

    def _C_tuple(m, t, a, r_tau_val):
        C_list = [r_tau_val]
        autres_operateurs = list(m.I)[1:] 
        for i in autres_operateurs:
            C_list.append(pyo.value(m.Rcomp[t, a, i]))
        return tuple(C_list)

    model.Cvec = pyo.Set(initialize=C_space)
    model.Sa = Sa_dict
    model.As = As_dict

    # --- PARAMÈTRES ---
    Zmax_data = [0, num_areas//10, num_areas//10, num_areas//10, num_areas//10, num_areas//10]
    model.Zmax = pyo.Param(model.T, initialize = Zmax_data)

    df_qa = pd.read_csv("STRATEGIC_GUIDELINES.csv", sep=";")
    QA_data = {row.TIME_SLOTS: row["QoE"] for _,row in df_qa.iterrows()}
    QA_data[0] = 0
    model.QA = pyo.Param(model.T, initialize = QA_data)

    ua0_data = {}
    for _, row in dA.iterrows(): 
        for col in dA.columns[1:]:
            if row["AREAS"] in A :
                ua0_data[row["AREAS"],col.split("-")[1], col] = row[col]
    
    for a in A:
        for i in I:
            for o in O:
                if (a,i,o) not in ua0_data:
                    ua0_data[a,i,o] = 0
                    
    model.ua0 = pyo.Param(model.A, model.I, model.O, initialize=ua0_data)

    df_dng = pd.read_csv("DEMAND.csv", sep=";")
    DNG_data = {row.TIME_SLOTS: row["5G"] for _,row in df_dng.iterrows()}
    DNG_data[0] = 0
    model.DNG = pyo.Param(model.T, initialize = DNG_data)

    df_capang = pd.read_csv("CAPACITY.csv", sep=";")
    CAPANG_data = {row.TIME_SLOTS: row["5G"] for _,row in df_capang.iterrows()}
    CAPANG_data[0] = 0
    model.CAPANG = pyo.Param(model.T, initialize = CAPANG_data)

    u_a_data = {}
    for _, row in dA.iterrows():
        if row["AREAS"] in A :
            somme = 0
            for col in dA.columns[1:]:
                somme += row[col]
            u_a_data[row["AREAS"]] = somme
    model.u_a = pyo.Param(model.A, initialize = u_a_data)

    df_Rcomp = pd.read_csv("COMPETITORS_STRATEGY.csv", sep=";")
    Rcomp_data = {}
    for _, row in df_Rcomp.iterrows():
        if row["AREAS"] in A :
            for col in df_Rcomp.columns[2:5]:
                Rcomp_data[row["TIME_SLOTS"],row["AREAS"], col] = 1 * row[col]
    model.Rcomp = pyo.Param(model.T, model.A, model.I, initialize=Rcomp_data)

    df_fdata = pd.read_csv("UPGRADE_FUNCTION.csv", sep=";")
    f_data = {}
    for a in A:
        for _, row in df_fdata.iterrows():
            C  = (row["ORANGE"], row["FREE MOBILE"], row["BOUYGUES TELECOM"], row["SFR"])
            O1 = row["OFFERS"] + "-" + row["FROM_OPERATOR"]
            O2 = "o5G-" + row["TO_OPERATOR"]
            f_data[(a, C, O1, O2)] = row["PERCENTAGES"]

    O_full = O 
    for a in A:
        for C in C_space:  
            for O1 in O_full:    
                for O2 in O_full:                                               
                    if (a, C, O1, O2) not in f_data:
                       f_data[a, C, O1, O2] = 0

    for a in A:
        for C in C_space:  
            for O1 in O_full:
                v = 1
                for O2 in O_full:
                    if O2 != O1 :
                        v -= f_data[a, C, O1, O2]
                f_data[a, C, O1, O1] = v

    model.f = pyo.Param(model.A, model.Cvec, model.O, model.O, initialize=f_data)

    L_data = {}
    U_data = {}
    eps = 1e-4

    for a in model.A:
        ua = pyo.value(model.u_a[a])
        for t in model.T:
            C0 = _C_tuple(model, t, a, 0)
            C1 = _C_tuple(model, t, a, 1)
            for o in model.O:
                coeffs = []
                for i_prev in model.I:
                    for o_prev in model.O_i[i_prev]:
                        d = pyo.value(model.f[a, C1, o_prev, o]) - pyo.value(model.f[a, C0, o_prev, o])
                        coeffs.append(d)
                d_max = max(coeffs)
                d_min = min(coeffs)
                U_data[(a, o, t)] = max(0.0, d_max) * ua + eps
                L_data[(a, o, t)] = min(0.0, d_min) * ua - eps

    model.U_sigma = pyo.Param(model.A, model.O, model.T, initialize=U_data, mutable=True)
    model.L_sigma = pyo.Param(model.A, model.O, model.T, initialize=L_data, mutable=True)

    # --- VARIABLES ---
    model.z = pyo.Var(model.T, model.S, within=pyo.Binary)
    model.r = pyo.Var(model.T, model.A, within=pyo.Binary)
    model.S_var = pyo.Var(model.T, model.A, model.I, model.O, within=pyo.Reals)
    model.u = pyo.Var(model.T, model.A, model.I, model.O, within=pyo.NonNegativeReals)
    model.u_site = pyo.Var(model.T, model.A, model.S, within=pyo.NonNegativeReals)

    # --- CONTRAINTES ---
    def coverage_upper(m, t, a):
        return m.r[t, a] <= sum(m.z[t, s] for s in Sa_dict[a])
    model.c_2 = pyo.Constraint(model.T, model.A, rule=coverage_upper)

    def coverage_lower(m, t, s, a):
        if a in As_dict[s]:
            return m.z[t, s] <= m.r[t, a]
        return pyo.Constraint.Skip
    model.c_3 = pyo.Constraint(model.T, model.S, model.A, rule=coverage_lower)

    def u_eq_rule(m, t, a, i, o):
        if o in O_i[i]:
            if t == min(m.T):
                return m.u[t, a, i, o] == m.ua0[a, i, o]
            C0 = _C_tuple(m, t, a, 0)
            migration_f0 = sum(m.f[a, C0, o_prev, o] * m.u[t-1, a, i_prev, o_prev] for i_prev in m.I for o_prev in m.O_i[i_prev])
            return m.u[t, a, i, o] == m.S_var[t, a, i, o] + migration_f0
        else:
            return m.u[t, a, i, o] == 0.0
    model.c_4 = pyo.Constraint(model.T, model.A, model.I, model.O, rule=u_eq_rule)

    def sigma_rule(m, t, a, i, o):
        if t == min(m.T):
            return 0.0
        C0 = _C_tuple(m, t, a, 0)
        C1 = _C_tuple(m, t, a, 1)
        return sum((m.f[a, C1, o_prev, o] - m.f[a, C0, o_prev, o]) * m.u[t-1, a, i_prev, o_prev] for i_prev in m.I for o_prev in m.O_i[i_prev])
    model.sigma = pyo.Expression(model.T, model.A, model.I, model.O, rule=sigma_rule)

    def c_6_rule(m, t, a, i, o):
        if t == min(m.T) or o not in O_i[i]:
            return pyo.Constraint.Skip
        U = m.U_sigma[a, o, t]
        return m.S_var[t, a, i, o] <= U * m.r[t, a]
    model.c_6 = pyo.Constraint(model.T, model.A, model.I, model.O, rule=c_6_rule)

    def c_7_rule(m, t, a, i, o):
        if o not in O_i[i]:
            return m.S_var[t, a, i, o] == 0.0
        if t == min(m.T):
            return m.S_var[t, a, i, o] == 0.0
        U = m.U_sigma[a, o, t]
        return m.S_var[t, a, i, o] <= m.r[t, a] * U
    model.c_7 = pyo.Constraint(model.T, model.A, model.I, model.O, rule=c_7_rule)

    def c_8_rule(m, t, a, i, o):
        if t == min(m.T) or o not in O_i[i]:
            return pyo.Constraint.Skip
        L = m.L_sigma[a, o, t]
        return m.S_var[t, a, i, o] <= m.sigma[t, a, i, o] - L * (1 - m.r[t, a])
    model.c_8_sigma = pyo.Constraint(model.T, model.A, model.I, model.O, rule=c_8_rule)

    def c_9_rule(m, t, a, i, o):
        if t == min(m.T) or o not in O_i[i]:
            return pyo.Constraint.Skip
        U = m.U_sigma[a, o, t]
        return m.S_var[t, a, i, o] >= m.sigma[t, a, i, o] - U * (1 - m.r[t, a])
    model.c_9 = pyo.Constraint(model.T, model.A, model.I, model.O, rule=c_9_rule)

    def assign_users(m, t, a):
        return m.u[t, a, τ, NG] == sum(m.u_site[t, a, s] for s in Sa_dict[a])
    model.c_10 = pyo.Constraint(model.T, model.A, rule=assign_users)

    def capacity(m, t, s):
        return sum(m.DNG[t] * m.u_site[t, a, s] for a in As_dict[s]) <= m.CAPANG[t] * m.z[t, s]
    model.c_11 = pyo.Constraint(model.T, model.S, rule=capacity)

    def limit_z(m, t):
        if t == min(m.T):
            return sum(m.z[t, s] for s in m.S) <= m.Zmax[t]
        return sum(m.z[t, s] - m.z[t-1, s] for s in m.S) <= m.Zmax[t]
    model.c_12 = pyo.Constraint(model.T, rule=limit_z)

    def cov_pop(m, t):
        return sum(m.u_a[a] * m.r[t, a] for a in m.A) >= m.QA[t] * sum(m.u_a[a] for a in m.A)
    model.c_13 = pyo.Constraint(model.T, rule=cov_pop)

    def growth_z(m, t, s):
        if t == min(m.T):
            return pyo.Constraint.Skip
        return m.z[t, s]>= m.z[t-1, s]
    model.c_growth_z = pyo.Constraint(model.T, model.S, rule=growth_z)

    # --- OBJECTIF ---
    def objective(m):
        T_end = max(m.T)
        return sum(m.u[T_end, a, τ, NG] for a in m.A)
    model.obj = pyo.Objective(rule=objective, sense=pyo.maximize)

    return model

In [5]:
from pyomo.opt import TerminationCondition

def generate_training_dataset(num_instances=30, areas_per_instance=20, solver_name=solver_):
    print(f"--- Génération du Dataset : {num_instances} instances ---")
    dataset_graphs = []
    solver = pyo.SolverFactory(solver_name)
    
    for i in range(num_instances):
        print(f"Génération et résolution de l'instance {i+1}/{num_instances}...")
        model_instance = build_pyomo_instance(seed_value=i, num_areas=areas_per_instance)
        results = solver.solve(model_instance, tee=False)
        
        if results.solver.termination_condition == TerminationCondition.optimal:
            graph_data = extract_graph_from_pyomo(model_instance)
            dataset_graphs.append(graph_data)
        else:
            print(f"Instance {i+1} non résolue de manière optimale, ignorée.")
            
    print(f"Dataset terminé : {len(dataset_graphs)} graphes valides générés.\n")
    return dataset_graphs

In [6]:
# --- RUN BLOCK ---
dataset = generate_training_dataset(num_instances=100, areas_per_instance=50)

if len(dataset) > 0:
    print("--- Début de l'entraînement du GNN ---")
    trained_model = train_gnn(dataset, epochs=100) 
    print("--- Entraînement terminé ---\n")
else:
    raise ValueError("Le dataset est vide.")

print("--- Création de l'instance de test (grande dimension) ---")


print("\n--- Processus complet terminé avec succès ! ---")

--- Génération du Dataset : 100 instances ---
Génération et résolution de l'instance 1/100...
Génération et résolution de l'instance 2/100...
Génération et résolution de l'instance 3/100...
Génération et résolution de l'instance 4/100...
Génération et résolution de l'instance 5/100...
Génération et résolution de l'instance 6/100...
Génération et résolution de l'instance 7/100...
Génération et résolution de l'instance 8/100...
Génération et résolution de l'instance 9/100...
Génération et résolution de l'instance 10/100...
Génération et résolution de l'instance 11/100...
Génération et résolution de l'instance 12/100...
Génération et résolution de l'instance 13/100...
Génération et résolution de l'instance 14/100...
Génération et résolution de l'instance 15/100...
Génération et résolution de l'instance 16/100...
Génération et résolution de l'instance 17/100...
Génération et résolution de l'instance 18/100...
Génération et résolution de l'instance 19/100...
Génération et résolution de l'in

In [33]:
test_model = build_pyomo_instance(seed_value=9, num_areas=200)

solved_model = solve_with_nlns(
    pyomo_model=test_model, 
    trained_gnn=trained_model, 
    solver_name=solver_, 
    fix_ratio=0.95
)


--- Démarrage du NLNS ---
NLNS : 2088 variables z[t,s] fixées par le GNN.
NLNS : Lancement du solveur classique sur les variables restantes...


RuntimeError: A feasible solution was not found, so no solution can be loaded. If using the appsi.solvers.Highs interface, you can set opt.config.load_solution=False. If using the environ.SolverFactory interface, you can set opt.solve(model, load_solutions = False). Then you can check results.termination_condition and results.best_feasible_objective before loading a solution.

In [ ]:
# import matplotlib.pyplot as plt
# import random

# def plot_random_sites_evolution(pyomo_model, num_sites=3):
#     """
#     Sélectionne aléatoirement un nombre défini de sites et affiche 
#     l'évolution de leur activation (z_s^t) sur un même graphique.
#     """
#     S_list = list(pyomo_model.S)
#     T_list = list(pyomo_model.T)
    
#     # Sécurité : on s'assure de ne pas demander plus de sites qu'il n'en existe
#     num_sites_to_plot = min(num_sites, len(S_list))
    
#     # Tirage au sort des sites
#     random_sites = random.sample(S_list, num_sites_to_plot)
    
#     plt.figure(figsize=(10, 5))
    
#     # Pour décaler très légèrement les courbes superposées (pour la lisibilité)
#     y_offset = 0.0 
    
#     for site in random_sites:
#         z_values = []
#         for t in T_list:
#             val = pyomo_model.z[t, site].value
#             # On récupère la valeur, et on applique le petit décalage visuel
#             z_values.append((round(val) if val is not None else 0) + y_offset)
            
#         # Tracé en escalier pour ce site
#         plt.step(T_list, z_values, where='post', marker='o', linewidth=2.5, alpha=0.7, label=f"Site {site}")
        
#         # On augmente le décalage pour le site suivant (évite que les lignes se cachent parfaitement)
#         y_offset += 0.02 

#     # Formatage visuel du graphique
#     plt.yticks([0, 1], ['Désactivé (0)', 'Activé (1)'])
#     plt.xticks(T_list)
#     plt.xlabel("Période de temps (t)")
#     plt.ylabel("État d'activation (z)")
#     plt.title(f"Évolution de l'activation pour {num_sites_to_plot} sites aléatoires")
#     plt.grid(axis='both', linestyle='--', alpha=0.5)
    
#     # Ajout de la légende pour différencier les sites
#     plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), title="Sites")
    
#     plt.tight_layout() # Ajuste les marges pour que la légende ne soit pas coupée
#     plt.show()

# # --- EXEMPLE D'UTILISATION ---
# # À lancer tout à la fin, une fois que 'solved_model' a été calculé.
# # Vous pouvez changer 'num_sites' pour en afficher 4, 5, etc.
# plot_random_sites_evolution(test_model, num_sites=1)

: 